# NumPy avanzado: lectura de archivos, formatos binarios y fechas

**Temas de la clase:**
1. Lectura de archivos con `np.genfromtxt`
2. Formato `.npy` / `.npz`: qué son y en qué se diferencian de un CSV
3. El parámetro `allow_pickle`: qué hace y por qué importa
4. `datetime` (Python) vs `numpy.datetime64`: diferencias y cuándo usar cada uno

> Notebook autocontenido: los archivos de ejemplo se generan en la primera sección, así que se puede correr de punta a punta sin depender de archivos externos.


In [ ]:
import numpy as np
import datetime
import os

# Carpeta de trabajo para los archivos de ejemplo
os.makedirs("data", exist_ok=True)
np.__version__


---
## 1. Lectura de archivos con `np.genfromtxt`

`genfromtxt` es la función de NumPy pensada para leer archivos de texto (CSV, TSV, etc.) que pueden tener:
- valores faltantes,
- columnas con distintos tipos de datos,
- encabezados,
- comentarios.

Es más lenta que `np.loadtxt`, pero mucho más robusta. La regla práctica es:

- **`np.loadtxt`**: datos limpios, homogéneos, sin faltantes.
- **`np.genfromtxt`**: datos "de la vida real", con huecos, texto y números mezclados, encabezados, etc.

Primero generamos un CSV de ejemplo con un dato faltante a propósito.


In [ ]:
csv_texto = """nombre,edad,altura,ciudad
Ana,23,1.68,CABA
Luis,,1.75,Rosario
Marta,34,1.60,CABA
Pedro,29,,Cordoba
"""

with open("data/personas.csv", "w") as f:
    f.write(csv_texto)

print(csv_texto)


### 1.1 Lectura básica

Si intentamos leer esto con `np.loadtxt` directamente, va a fallar: hay texto (nombres, ciudades) y valores faltantes. `genfromtxt` sí puede manejarlo.


In [ ]:
datos = np.genfromtxt(
    "data/personas.csv",
    delimiter=",",
    names=True,      # usa la primera fila como nombres de columna
    dtype=None,      # infiere el tipo de cada columna
    encoding="utf-8"
)

datos


In [ ]:
# Es un array estructurado: cada fila es un "registro" con campos con nombre
print(datos.dtype)
print(datos["nombre"])
print(datos["edad"])


Noten los valores faltantes: como no especificamos qué hacer con ellos, `genfromtxt` intenta inferir el tipo y, para las columnas numéricas, reemplaza el vacío por `nan`. Para las columnas de texto, hay que ser explícitos.


### 1.2 Parámetros clave

| Parámetro | Para qué sirve |
|---|---|
| `delimiter` | separador de columnas (`,`, `;`, `\t`, etc.) |
| `skip_header` / `skip_footer` | cuántas líneas ignorar al principio/final |
| `names` | `True` para tomar la primera fila como nombres, o una lista de nombres |
| `dtype` | tipo de dato; `None` para que infiera columna por columna |
| `missing_values` | qué strings considerar "faltante" (ej: `"NA"`, `"?"`, `""`) |
| `filling_values` | con qué valor rellenar los faltantes |
| `usecols` | qué columnas leer |
| `comments` | caracter que marca comentarios a ignorar (por defecto `#`) |


In [ ]:
# Ejemplo controlando faltantes explícitamente y quedándonos solo con columnas numéricas
datos_num = np.genfromtxt(
    "data/personas.csv",
    delimiter=",",
    skip_header=1,
    usecols=(1, 2),          # edad, altura
    missing_values="",
    filling_values=-1,       # relleno explícito para faltantes
    dtype=float
)

datos_num


### 1.3 `genfromtxt` vs `loadtxt`: comparación directa

Vamos a generar un CSV **sin faltantes y sin texto**, apto para `loadtxt`, y comparar tiempos y comportamiento.


In [ ]:
import time

# CSV grande, limpio, todo numérico
n = 200_000
matriz = np.random.rand(n, 4)
np.savetxt("data/matriz_limpia.csv", matriz, delimiter=",")

t0 = time.time()
a = np.loadtxt("data/matriz_limpia.csv", delimiter=",")
t1 = time.time()
b = np.genfromtxt("data/matriz_limpia.csv", delimiter=",")
t2 = time.time()

print(f"loadtxt:    {t1 - t0:.3f} s")
print(f"genfromtxt: {t2 - t1:.3f} s")
print("¿Resultados iguales?", np.allclose(a, b))


**Conclusión para mostrar en clase:** con datos limpios `loadtxt` es notablemente más rápido. `genfromtxt` paga un costo extra porque analiza fila por fila buscando problemas (faltantes, tipos mixtos, etc.). La elección es un trade-off entre **velocidad** y **robustez**.


---
## 2. El formato `.npy` (y `.npz`)

Hasta ahora leímos texto plano. NumPy tiene su **propio formato binario**: `.npy` para un solo array, `.npz` para varios arrays juntos (opcionalmente comprimidos).

### ¿Por qué usar `.npy` en vez de CSV?

| | CSV / texto | `.npy` |
|---|---|---|
| Legible por humanos | Sí | No (binario) |
| Velocidad de lectura/escritura | Más lenta | Mucho más rápida |
| Tamaño en disco | Mayor | Menor (y `.npz` comprimido, aún menor) |
| Preserva `dtype` y `shape` exactos | No (hay que reconstruirlo) | Sí, exacto |
| Portabilidad entre lenguajes | Alta | Baja (pensado para NumPy) |


In [ ]:
arr = np.random.rand(1_000_000)

# Guardar y comparar tamaños/tiempos: texto vs binario
np.savetxt("data/arr.csv", arr, delimiter=",")
np.save("data/arr.npy", arr)

import os
print("Tamaño CSV: ", os.path.getsize("data/arr.csv") / 1e6, "MB")
print("Tamaño NPY: ", os.path.getsize("data/arr.npy") / 1e6, "MB")

t0 = time.time()
_ = np.loadtxt("data/arr.csv", delimiter=",")
t1 = time.time()
_ = np.load("data/arr.npy")
t2 = time.time()

print(f"Leer CSV: {t1-t0:.4f} s")
print(f"Leer NPY: {t2-t1:.4f} s")


### 2.1 `.npz`: varios arrays en un solo archivo

Con `np.savez` (sin comprimir) o `np.savez_compressed` (comprimido) podemos guardar varios arrays juntos, cada uno con su nombre.


In [ ]:
edades = np.array([23, 31, 29, 40])
alturas = np.array([1.68, 1.75, 1.60, 1.81])

np.savez("data/personas.npz", edad=edades, altura=alturas)
np.savez_compressed("data/personas_comp.npz", edad=edades, altura=alturas)

cargado = np.load("data/personas.npz")
print(list(cargado.keys()))
print(cargado["edad"])
print(cargado["altura"])


In [ ]:
print("Tamaño npz sin comprimir:", os.path.getsize("data/personas.npz"), "bytes")
print("Tamaño npz comprimido:      ", os.path.getsize("data/personas_comp.npz"), "bytes")


**Punto para remarcar en clase:** para datasets chicos la diferencia es anecdótica, pero en datasets grandes (millones de filas, arrays multidimensionales) la diferencia de tamaño y velocidad se vuelve significativa. La regla general:

- **Compartir datos / abrir en Excel / otro lenguaje** → CSV u otro formato de texto.
- **Guardar resultados intermedios de un pipeline en Python/NumPy** → `.npy` / `.npz`.


---
## 3. `allow_pickle`

Cuando un array de NumPy tiene `dtype=object` (por ejemplo, contiene listas, diccionarios, strings de longitud variable mezclados con otros tipos, etc.), NumPy no puede serializarlo con su formato binario simple. En esos casos usa **pickle** de Python por debajo.

Desde NumPy 1.16.3, por defecto `np.load` tiene `allow_pickle=False`. Es una medida de seguridad.


In [ ]:
# Un array de tipo "object": cada elemento puede ser cualquier cosa (acá, listas de distinto largo)
arr_objetos = np.array([[1, 2], [3, 4, 5], "texto"], dtype=object)
np.save("data/objetos.npy", arr_objetos)  # esto internamente usa pickle

# Intentar cargarlo SIN permitir pickle -> falla
try:
    np.load("data/objetos.npy", allow_pickle=False)
except ValueError as e:
    print("Error esperado:")
    print(e)


In [ ]:
# Con allow_pickle=True sí funciona
recargado = np.load("data/objetos.npy", allow_pickle=True)
recargado


### ¿Por qué es un riesgo de seguridad?

`pickle` no solo serializa datos: puede ejecutar código arbitrario al deserializar un objeto malicioso. Esto significa que **cargar un `.npy`/`.npz` con `allow_pickle=True` desde una fuente no confiable es potencialmente tan peligroso como ejecutar un script desconocido**.

**Buenas prácticas para la clase:**
- Dejar `allow_pickle=False` (el default) salvo que sepas con certeza que el archivo es propio y confiable.
- Si un array requiere `dtype=object`, evaluar si en realidad conviene reestructurar los datos (por ejemplo, usar un array estructurado con campos con nombre, como vimos en la Sección 1) en lugar de depender de pickle.
- Nunca cargar con `allow_pickle=True` un archivo `.npy`/`.npz` descargado de una fuente externa no verificada.


---
## 4. `datetime.datetime` (Python) vs `numpy.datetime64`

### 4.1 `datetime` de Python (módulo estándar)

Es un objeto rico en funcionalidad (zonas horarias, formateo, aritmética de calendario), pero:
- Cada `datetime` es un **objeto Python independiente**.
- Un array de NumPy con `datetime` de Python queda como `dtype=object` → pierde las ventajas de vectorización de NumPy (es lento, ocupa más memoria).


In [ ]:
fechas_py = [datetime.datetime(2024, 1, 1), datetime.datetime(2024, 1, 2), datetime.datetime(2024, 1, 3)]
arr_datetime_py = np.array(fechas_py)
print(arr_datetime_py.dtype)   # object!
arr_datetime_py


### 4.2 `numpy.datetime64`

Es un **tipo de dato nativo de NumPy**, tan eficiente como un `int64` o `float64`. Se puede indicar la precisión: años (`Y`), meses (`M`), días (`D`), horas (`h`), minutos (`m`), segundos (`s`), nanosegundos (`ns`), etc.


In [ ]:
fecha = np.datetime64("2024-01-01")
print(fecha, fecha.dtype)

fechas_np = np.array(["2024-01-01", "2024-01-02", "2024-01-03"], dtype="datetime64[D]")
print(fechas_np.dtype)
fechas_np


### 4.3 Comparación práctica: vectorización y performance

La gran diferencia se nota al hacer operaciones sobre muchas fechas.


In [ ]:
n = 100_000

# --- Con datetime.datetime (Python puro) ---
base = datetime.datetime(2024, 1, 1)
fechas_python = [base + datetime.timedelta(days=i) for i in range(n)]

t0 = time.time()
diffs_python = [f - base for f in fechas_python]   # loop en Python puro
t1 = time.time()

# --- Con numpy.datetime64 (vectorizado) ---
fechas_numpy = np.datetime64("2024-01-01") + np.arange(n).astype("timedelta64[D]")

t0b = time.time()
diffs_numpy = fechas_numpy - np.datetime64("2024-01-01")   # operación vectorizada
t1b = time.time()

print(f"datetime.datetime (loop Python): {t1 - t0:.4f} s")
print(f"numpy.datetime64  (vectorizado): {t1b - t0b:.6f} s")


### 4.4 Aritmética y unidades

`numpy.datetime64` trabaja junto con `numpy.timedelta64` para diferencias de tiempo, y respeta la unidad de precisión declarada.


In [ ]:
d1 = np.datetime64("2024-03-01")
d2 = np.datetime64("2024-01-15")

diferencia = d1 - d2
print(diferencia, diferencia.dtype)   # timedelta64[D]

# Cambiar de unidad
print(diferencia.astype("timedelta64[h]"))  # a horas


In [ ]:
# Generar rangos de fechas: el equivalente numpy de un "date_range"
rango = np.arange("2024-01-01", "2024-01-10", dtype="datetime64[D]")
rango


### 4.5 Cuándo usar cada uno

| Necesitás... | Usá |
|---|---|
| Trabajar con **muchas** fechas, hacer cuentas rápido, filtrar/vectorizar | `numpy.datetime64` |
| Zonas horarias, formateo local, lógica de calendario compleja (día de la semana, meses, feriados) | `datetime.datetime` (o `pandas.Timestamp`, que combina ambos mundos) |
| Guardar fechas dentro de un array estructurado o una columna de un dataset grande | `numpy.datetime64` |

**Nota para la clase:** en la práctica, cuando se usa `pandas`, esto queda resuelto: `pandas.Timestamp` combina la riqueza de `datetime` con la eficiencia interna de `datetime64`. Pero entender la diferencia en NumPy puro ayuda a entender qué pasa "debajo" de pandas.


---
## Resumen de la clase

- **`genfromtxt`**: para leer texto con datos faltantes, tipos mixtos o encabezados. Más lento pero más robusto que `loadtxt`.
- **`.npy` / `.npz`**: formato binario propio de NumPy. Mucho más rápido y compacto que texto, pero no es legible por humanos ni fácilmente portable a otros lenguajes.
- **`allow_pickle`**: necesario para arrays `dtype=object`; representa un riesgo de seguridad si se cargan archivos de origen no confiable con `allow_pickle=True`.
- **`datetime` vs `datetime64`**: `datetime` es rico pero no vectorizable dentro de NumPy (queda como `object`); `datetime64` es nativo, rápido y vectorizado, ideal para grandes volúmenes de fechas.


---
## Ejercicios

Cinco ejercicios, uno por cada bloque de la clase (y uno integrador al final). Cada uno tiene una celda de código vacía para resolver en vivo.


### Ejercicio 1 — `genfromtxt` con faltantes explícitos

Crear (con `open`/`write`, como hicimos con `personas.csv`) un archivo `productos.csv` con las columnas `producto`, `precio`, `stock`, dejando **al menos 2 valores faltantes** (uno numérico y representado con la cadena `"NA"`, no con una celda vacía).

Leerlo con `np.genfromtxt` indicando explícitamente:
- `missing_values="NA"`
- `filling_values=0`
- `names=True`

Mostrar el array resultante y el `dtype`.


In [ ]:
# Ejercicio 1: tu código acá



### Ejercicio 2 — `loadtxt` vs `genfromtxt`: ¿cuándo falla uno y no el otro?

1. Generar un CSV limpio y numérico de 500.000 filas x 3 columnas (como hicimos con `matriz_limpia.csv`).
2. Leerlo con `loadtxt` y con `genfromtxt`, midiendo el tiempo de cada uno con `time.time()`.
3. Ahora **rompan** una sola celda del archivo (reemplacen un número por la palabra `"error"` con un editor de texto o reescribiendo el archivo). Intenten leerlo de nuevo con `loadtxt` y con `genfromtxt`.

Respondan en una celda markdown: ¿cuál de los dos falla? ¿Cuál seguiría funcionando (aunque sea con un `nan`)?


In [ ]:
# Ejercicio 2: tu código acá



### Ejercicio 3 — Comparar formatos de guardado

Tomar el array estructurado que generaron en el Ejercicio 1 (o el de `personas.csv` de la clase) y guardarlo de 3 formas distintas:
1. Como texto con `np.savetxt` (ojo: para arrays estructurados van a necesitar convertir o usar `fmt`).
2. Como `.npy` con `np.save`.
3. Como `.npz` comprimido con `np.savez_compressed`.

Comparar el tamaño en disco de los 3 archivos con `os.path.getsize` y armar una tabla (puede ser un `print` prolijo) con los resultados.


In [ ]:
# Ejercicio 3: tu código acá



### Ejercicio 4 — `allow_pickle` y seguridad

1. Crear un array `dtype=object` que mezcle un diccionario, una lista y un string (por ejemplo `np.array([{"a":1}, [1,2,3], "hola"], dtype=object)`).
2. Guardarlo con `np.save`.
3. Intentar cargarlo con `allow_pickle=False` y capturar el error con `try/except` (como hicimos en la clase).
4. Cargarlo correctamente con `allow_pickle=True`.

En una celda markdown, explicar con sus palabras: si alguien les manda por mail un archivo `.npy` y no saben de dónde salió, ¿por qué NO deberían cargarlo con `allow_pickle=True` sin pensarlo?


In [ ]:
# Ejercicio 4: tu código acá



### Ejercicio 5 — Integrador: `datetime64` + análisis vectorizado

1. Generar todas las fechas de un mes completo (por ejemplo, agosto de 2026) como un array `datetime64[D]` usando `np.arange`.
2. Sin usar ningún `for`, calcular el día de la semana de cada fecha. *Pista:* `np.datetime64` tiene una relación fija con los días de la semana si se convierte a un tipo entero de días desde el "epoch"; investiguen `astype('datetime64[D]').astype(int) % 7` y comparen contra un `datetime.date.weekday()` de Python para un par de fechas, para verificar el offset correcto.
3. Contar cuántos días del mes caen en fin de semana usando una máscara booleana (indexado vectorizado, sin loops).
4. Comparar cuánto tardarían en hacer el mismo cálculo con una lista de `datetime.date` de Python y un `for`, usando `time.time()`.

Cerrar con una celda markdown: ¿qué ventaja concreta les dio `datetime64` acá frente al enfoque con `datetime` + loop?


In [ ]:
# Ejercicio 5: tu código acá

